# Layer One
*The first neural network in the Deep Move collection — a CNN trained on grandmaster games via supervised learning.*

---

## 1. Introduction
- What is Layer One?
- Goal: predict the next move given a board position
- Tech stack: PyTorch, python-chess, Lichess Elite Database
- Architecture summary in one sentence

---

### Raw Data Format

Our training data comes from the [Lichess Elite Database](https://database.nikonoel.fr/) — games between players rated 2200+ on Lichess. Each game is stored in PGN (Portable Game Notation) format:

```
[Event "Rated Blitz game"]
[LichessURL "https://lichess.org/6gqOPa4x"]
[Date "2025.10.01"]
[White "hhcuuid"]
[Black "CyrilMarcelin"]
[Result "1-0"]
[WhiteTitle "GM"]
[BlackTitle "GM"]
[WhiteElo "2602"]
[BlackElo "2616"]
[ECO "A40"]
[Opening "English Defense"]
[TimeControl "180+0"]

1. d4 b6 2. c4 Bb7 3. Nc3 g6 4. e4 Bg7 5. Nf3 e6 6. g3 Ne7
7. Bg2 d6 8. O-O O-O 9. Be3 Nd7 10. Qd2 Re8 11. Bh6 Bh8
12. Rfe1 e5 13. Rad1 Nc6 14. Bg5 Bf6 15. Bxf6 Nxf6 16. Nd5 Nd7
17. b3 Kg7 18. Bh3 Nf8 19. dxe5 dxe5 20. Qc3 Qb8 21. Qd2 Bc8
22. Qg5 Bxh3 23. Qf6+ Kg8 24. Qxc6 Re6 25. Qxc7 Bg4 26. Ng5 Qe8
27. f3 Rc8 28. Nxe6 1-0
```

**What we're looking at:**
- **Metadata** — player names, titles (both GMs), ELO ratings (2602 vs 2616), opening, time control
- **Move sequence** — 28 moves of a Blitz game in standard algebraic notation
- **Result** — `1-0` (White wins)

Each position in this game becomes a training sample: the board state is the input, and the move played is the target output.

## 2. Data Pipeline
- Data source and filtering criteria
- PGN parsing process
- Board → tensor conversion (with visual example)
- Move → index encoding (with visual example)
- Train/validation/test split

---

### Imports & Board Representation

We use `python-chess` for game parsing and move validation, and `numpy` for tensor operations.

The board is represented as a **12×8×8 tensor** — 12 binary planes, one per piece type and color. `PIECE_TO_PLANE` maps each (piece, color) pair to its corresponding plane index. `board_to_tensor()` takes any board position and returns this tensor.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install python-chess

In [5]:
import chess
import chess.pgn
import numpy as np

# Piece type to plane index mapping
PIECE_TO_PLANE = {
    (chess.PAWN, chess.WHITE): 0,
    (chess.KNIGHT, chess.WHITE): 1,
    (chess.BISHOP, chess.WHITE): 2,
    (chess.ROOK, chess.WHITE): 3,
    (chess.QUEEN, chess.WHITE): 4,
    (chess.KING, chess.WHITE): 5,
    (chess.PAWN, chess.BLACK): 6,
    (chess.KNIGHT, chess.BLACK): 7,
    (chess.BISHOP, chess.BLACK): 8,
    (chess.ROOK, chess.BLACK): 9,
    (chess.QUEEN, chess.BLACK): 10,
    (chess.KING, chess.BLACK): 11,
}

def board_to_tensor(board):
    """
    Converts a python-chess Board object into a 12x8x8 numpy tensor.

    Each of the 12 planes represents one piece type and color:
        0: White Pawns     6: Black Pawns
        1: White Knights   7: Black Knights
        2: White Bishops   8: Black Bishops
        3: White Rooks     9: Black Rooks
        4: White Queen    10: Black Queen
        5: White King     11: Black King

    Each plane is an 8x8 grid where 1 indicates the piece is on that
    square and 0 indicates it is not.

    Args:
        board: a chess.Board object representing the current position.

    Returns:
        np.ndarray of shape (12, 8, 8) with binary values.
    """
    
    tensor = np.zeros((12, 8, 8), dtype=np.uint8)
    for (piece_type, color), plane in PIECE_TO_PLANE.items():
        for square in board.pieces(piece_type, color):
            row = square // 8
            col = square % 8
            tensor[plane][row][col] = 1
    return tensor

### Data Pipeline

Reads PGN games from the Lichess Elite Database and converts them into numpy arrays ready for training. Each board position becomes a 12×8×8 tensor (input), and each move becomes an index from 0-4095 encoding the from-square and to-square (target).

The first 1k games are reserved for validation, the next 5k for testing, and the rest for training. Files are saved as `.npy` for fast loading.

In [6]:
from tqdm import tqdm


def transform_game(limit, pgn):

    """
    Reads up to 'limit' games from the Lichess Elite PGN file and
    splits them into training, validation, and test sets.

    Split:
        - Games 0-999 (1k games): validation set
        - Games 1000-5999 (5k games): test set
        - Games 6000+: training set

    For each game, every position is converted to a 12x8x8 tensor
    and the corresponding move is encoded as a single index
    (from_square * 64 + to_square).

    Saves 6 .npy files to data/formatted_data/:
        X-{limit}-val.npy, Y-{limit}-val.npy
        X-{limit}-test.npy, Y-{limit}-test.npy
        X-{limit}-train.npy, Y-{limit}-train.npy

    Args:
        limit: number of games to process.
    """

    
    

    X = []  # board states
    Y = []  # move indices
    X_test = []
    Y_test = []
    X_val = []
    Y_val= []



    for i in tqdm(range(limit)):

        game = chess.pgn.read_game(pgn)
        if game is None:
            break

        if i < 1000:

            board = game.board()
            for move in game.mainline_moves():
                X_val.append(board_to_tensor(board))
                Y_val.append(move.from_square * 64 + move.to_square)
                board.push(move)

        elif i < 6000:
 
            board = game.board()
            for move in game.mainline_moves():
                X_test.append(board_to_tensor(board))
                Y_test.append(move.from_square * 64 + move.to_square)
                board.push(move)
    
        else:

            board = game.board()
            for move in game.mainline_moves():
                X.append(board_to_tensor(board))
                Y.append(move.from_square * 64 + move.to_square)
                board.push(move)




    X_val = np.array(X_val)
    Y_val = np.array(Y_val)
    np.save(f"data/formatted_data/{limit//1000}k/X-{limit//1000}k-val.npy", X_val)
    np.save(f"data/formatted_data/{limit//1000}k/Y-{limit//1000}k-val.npy", Y_val)

    X_test = np.array(X_test)
    Y_test = np.array(Y_test)
    np.save(f"data/formatted_data/{limit//1000}k/X-{limit//1000}k-test.npy", X_test)
    np.save(f"data/formatted_data/{limit//1000}k/Y-{limit//1000}k-test.npy", Y_test)

    X = np.array(X)
    Y = np.array(Y)
    np.save(f"data/formatted_data/{limit//1000}k/X-{limit//1000}k-train.npy", X)
    np.save(f"data/formatted_data/{limit//1000}k/Y-{limit//1000}k-train.npy", Y)


def transform_game_max(pgns):
    drive_path = "/content/drive/MyDrive/chessbot"
    
    X_val = []
    Y_val = []
    X_test = []
    Y_test = []
    X = []
    Y = []
    
    i = 0
    chunk_num = 0
    
    for pgn in pgns:
        while True:
            game = chess.pgn.read_game(pgn)
            if game is None:
                break
            
            if i < 1000:
                board = game.board()
                for move in game.mainline_moves():
                    X_val.append(board_to_tensor(board))
                    Y_val.append(move.from_square * 64 + move.to_square)
                    board.push(move)
            elif i < 6000:
                board = game.board()
                for move in game.mainline_moves():
                    X_test.append(board_to_tensor(board))
                    Y_test.append(move.from_square * 64 + move.to_square)
                    board.push(move)
            else:
                board = game.board()
                for move in game.mainline_moves():
                    X.append(board_to_tensor(board))
                    Y.append(move.from_square * 64 + move.to_square)
                    board.push(move)
                
                # Save chunk every 100k games
                if (i - 6000) % 50000 == 49999:
                    chunk_num += 1
                    np.save(f"{drive_path}/X-max-train-chunk{chunk_num}.npy", np.array(X))
                    np.save(f"{drive_path}/Y-max-train-chunk{chunk_num}.npy", np.array(Y))
                    print(f"Saved chunk {chunk_num} — {len(X)} positions")
                    X = []
                    Y = []
            
            i += 1
            if i % 10000 == 0:
                print(f"Processed {i} games")
    
    # Save remaining training data
    if len(X) > 0:
        chunk_num += 1
        np.save(f"{drive_path}/X-max-train-chunk{chunk_num}.npy", np.array(X))
        np.save(f"{drive_path}/Y-max-train-chunk{chunk_num}.npy", np.array(Y))
        print(f"Saved chunk {chunk_num} — {len(X)} positions")
    
    # Save val and test
    np.save(f"{drive_path}/X-max-val.npy", np.array(X_val))
    np.save(f"{drive_path}/Y-max-val.npy", np.array(Y_val))
    np.save(f"{drive_path}/X-max-test.npy", np.array(X_test))
    np.save(f"{drive_path}/Y-max-test.npy", np.array(Y_test))
    
    print(f"Done! Total games: {i}, Total chunks: {chunk_num}")

    


### Process 100k Games (Control Dataset)

Generates the control dataset — 1k val, 5k test, 94k training games (~8.3M positions).

In [32]:
pgn = open("data/raw_data/lichess_elite_2025-10.pgn")
transform_game(100000, pgn)

100%|██████████| 100000/100000 [03:07<00:00, 532.22it/s]


### Verify Output

Quick sanity check — load the training data and inspect shapes and a sample position to confirm the pipeline worked correctly.

In [35]:
X = np.load("data/formatted_data/100k/X-100k-train.npy")
Y = np.load("data/formatted_data/100k/Y-100k-train.npy")

print(X.shape)  # should be (N, 12, 8, 8)
print(Y.shape)  # should be (N,)

print(X[51][0])  # white pawns plane of the first position
print(X[51][6])  # white pawns plane of the first position

(8368614, 12, 8, 8)
(8368614,)
[[0 0 0 0 0 0 0 0]
 [1 1 1 0 0 1 0 1]
 [0 0 0 1 0 0 1 0]
 [0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 1 1 0 0 0 0 0]
 [1 0 0 0 1 0 0 0]
 [0 0 0 1 0 1 1 1]
 [0 0 0 0 0 0 0 0]]


### Process 50k Games (Size variation)

Generates 50k games, part of the training size model variation

In [4]:
pgn = open("data/raw_data/lichess_elite_2025-10.pgn")
transform_game(50000, pgn)

100%|██████████| 50000/50000 [01:34<00:00, 527.73it/s]


### Process 200k Games (Size variation)

Generates 200k games, part of the training size model variation

In [5]:
pgn = open("data/raw_data/lichess_elite_2025-10.pgn")
transform_game(200000, pgn)

100%|██████████| 200000/200000 [06:34<00:00, 507.17it/s]


### Process ~560k Games (Size variation)

Generates ~560k games, part of the training size model variation

In [6]:
import os
os.listdir("/content/drive/MyDrive/chessbot")

['lichess_elite_2025-11.pgn', 'lichess_elite_2025-10.pgn', '560k']

In [7]:
!cp "/content/drive/MyDrive/chessbot/lichess_elite_2025-10.pgn" /content/
!cp "/content/drive/MyDrive/chessbot/lichess_elite_2025-11.pgn" /content/

pgn1 = open("/content/drive/MyDrive/chessbot/lichess_elite_2025-10.pgn")
pgn2 = open("/content/drive/MyDrive/chessbot/lichess_elite_2025-11.pgn")
pgns = [pgn1, pgn2]
transform_game_max(pgns)

Processed 10000 games
Processed 20000 games
Processed 30000 games
Processed 40000 games
Processed 50000 games
Saved chunk 1 — 4462624 positions
Processed 60000 games
Processed 70000 games
Processed 80000 games
Processed 90000 games
Processed 100000 games
Saved chunk 2 — 4438312 positions
Processed 110000 games
Processed 120000 games
Processed 130000 games
Processed 140000 games
Processed 150000 games
Saved chunk 3 — 4425911 positions
Processed 160000 games
Processed 170000 games
Processed 180000 games
Processed 190000 games
Processed 200000 games
Saved chunk 4 — 4423045 positions
Processed 210000 games
Processed 220000 games
Processed 230000 games
Processed 240000 games
Processed 250000 games
Saved chunk 5 — 4401416 positions
Processed 260000 games
Processed 270000 games
Processed 280000 games
Processed 290000 games
Processed 300000 games
Saved chunk 6 — 4425476 positions
Processed 310000 games
Processed 320000 games
Processed 330000 games
Processed 340000 games
Processed 350000 games
